# NCBI Entrez API 同源序列搜索

本 Notebook 演示：
- 用 Biopython 调用 NCBI Entrez API
- ESearch + ESummary 获取同源序列摘要
- 输出 Top5 相似序列到 DataFrame

In [ ]:
from Bio import Entrez
import pandas as pd

Entrez.email = "yumengzhang77@gmail.com"
Entrez.api_key = "你的API密钥"

In [ ]:
# Day 5
def search_similar(sequence, top_n=5):
    search_term = sequence[:20]
    handle = Entrez.esearch(db="protein", term=search_term, retmax=top_n)
    record = Entrez.read(handle)
    handle.close()
    
    id_list = record["IdList"]
    if not id_list:
        return pd.DataFrame()
    
    ids_str = ",".join(id_list)
    handle = Entrez.esummary(db="protein", id=ids_str)
    records = Entrez.read(handle)
    handle.close()
    
    data = []
    for summary in records:
        data.append({
            "Accession": summary.get("AccessionVersion", "N/A"),
            "Species": summary.get("Organism", "N/A"),
            "Length": summary.get("Length", "N/A"),
            "Title": summary.get("Title", "N/A")[:80],
        })
    return pd.DataFrame(data)

# 测试
df_hits = search_similar(seqs[0]["seq"])
df_hits

## 小结
通过 NCBI Entrez API 实现了结构化获取公共数据库序列数据的能力。API 调用与公司后端的 HTTP 请求模式完全一致。